# 05-09 Binding: 모델에 파라미터와 도구 미리 연결하기

체인을 실행할 때마다 같은 파라미터를 반복해서 전달하는 건 번거로워요. `bind()` 메서드를 사용하면 체인 **구성 시점**에 파라미터를 미리 고정할 수 있어요.

파이썬의 `functools.partial()`과 비슷한 개념이에요. 함수의 일부 인자를 미리 채워두고, 나머지만 나중에 전달하는 거예요.

## 학습 목표

- `bind(stop=...)`로 특정 토큰에서 생성을 중단시켜 출력 길이를 제어할 수 있어요
- `bind(tools=[...])`로 OpenAI 도구(Tool) 스키마를 모델에 연결하고 구조화된 응답을 얻을 수 있어요
- `bind()`로 고정된 파라미터와 `invoke()` 시 전달하는 파라미터의 차이를 설명할 수 있어요

## 사전 지식

- Ch01의 `ChatOpenAI` 기본 호출 방법
- LCEL 파이프 연산자(`|`) 사용법

---

> **bind() 동작 흐름** — 아래 다이어그램은 `bind()`로 고정한 파라미터가 체인 실행 시 어떻게 전달되는지 보여줘요.

```mermaid
flowchart LR
    subgraph 체인 구성 시점
        B["bind()<br/>stop='SOLUTION'<br/>tools=[...]"]:::input
    end

    subgraph 실행 시점
        INV["invoke()<br/>{'question': '...'}"]:::process
    end

    M["모델<br/>ChatOpenAI"]:::process

    B -->|"고정 파라미터 전달"| M
    INV -->|"런타임 파라미터 전달"| M
    M --> R["응답"]:::output

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404
```

> 🔑 **핵심 개념**: `bind()`는 체인의 특정 단계에 "고정 파라미터"를 미리 설정하는 메서드예요. `functools.partial()`과 비슷해요.

### bind() vs invoke() 파라미터 전달 비교

| | `bind()` | `invoke()` |
|---|---|---|
| **시점** | 체인 구성 시 (한 번만) | 체인 실행 시 (매번) |
| **용도** | 고정 설정 (stop, tools, temperature) | 동적 입력 (질문, 문서 등) |
| **코드 위치** | 체인 정의 부분 | 체인 호출 부분 |

**주요 활용:**
- Stop 토큰(Stop Token) 설정
- OpenAI Tools 연결
- 기타 모델 파라미터 고정 (temperature, max_tokens 등)

In [1]:
# API 키를 환경변수로 관리하기 위한 설정 파일
from dotenv import load_dotenv

# API 키 정보 로드
load_dotenv()


True

In [2]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI

# 모델 초기화
model = ChatOpenAI(model_name="gpt-4o-mini", temperature=0)


## 1. Stop 토큰 바인딩

### Stop 토큰이란?

Stop 토큰(Stop Token)은 LLM에게 "여기서 생성을 멈춰"라고 지시하는 문자열이에요. LLM은 출력 중에 이 문자열을 만나면 즉시 생성을 중단해요.

예를 들어, 프롬프트가 `EQUATION:` 과 `SOLUTION:` 두 섹션으로 구성된 응답을 유도한다면, `stop="SOLUTION"`을 설정하면 방정식 부분만 받을 수 있어요.

> 🎯 **강의 포인트**: `bind(stop=...)` 패턴은 구조화된 프롬프트(EQUATION:/SOLUTION: 형식 등)와 함께 쓸 때 효과적이에요. stop 토큰 이전까지만 출력을 받으면 불필요한 토큰 비용을 절감할 수 있어요.

> 💡 **실무 팁**: stop 토큰은 배포 환경에서 출력 길이를 엄격하게 제어해야 할 때 유용해요. `max_tokens`와 달리 의미 단위로 끊을 수 있어서 더 정밀한 제어가 가능해요.

In [3]:
# ============================================================
# TODO: model.bind(stop="SOLUTION")으로 stop 토큰을 바인딩하고 기본 체인과 비교하세요
# 힌트: model.bind(stop="SOLUTION") — 이 문자열이 나타나면 생성 중단
# 예상 결과: stop 토큰 바인딩 체인은 "EQUATION: ..." 부분만 출력
# ============================================================

# ---------------------------------------------------
# bind(stop=...): stop 토큰으로 생성 길이 제어
# ---------------------------------------------------

# 1단계: 프롬프트 정의
# EQUATION:, SOLUTION: 형식으로 출력하도록 지시
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "대수 기호를 사용하여 다음 방정식을 작성한 다음 풀이하세요. "
            "다음 형식을 사용하세요:\n\nEQUATION:...\nSOLUTION:...\n\n",
        ),
        ("human", "{equation_statement}"),
    ]
)

# 2단계: 기본 체인 (stop 토큰 없음 — 전체 출력)
basic_chain = (
    {"equation_statement": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

result_basic = basic_chain.invoke("x의 세제곱에 7을 더하면 12와 같다")

print("=" * 60)
print("📥 기본 체인 실행 (stop 토큰 없음)")
print("=" * 60)
print(result_basic)


📥 기본 체인 실행 (stop 토큰 없음)
EQUATION: \( x^3 + 7 = 12 \)

SOLUTION: 
1. 양변에서 7을 뺍니다:
   \[
   x^3 = 12 - 7
   \]
   \[
   x^3 = 5
   \]

2. 양변의 세제곱근을 구합니다:
   \[
   x = \sqrt[3]{5}
   \]

따라서, \( x = \sqrt[3]{5} \)입니다.


In [4]:
# Stop 토큰을 바인딩한 체인
# 
# "SOLUTION" 토큰에서 생성이 중단됨

chain_with_stop = (
    {"equation_statement": RunnablePassthrough()}
    | prompt
    | model.bind(stop="SOLUTION")  # stop 토큰 바인딩
    | StrOutputParser()
)

result_stop = chain_with_stop.invoke("x의 세제곱에 7을 더하면 12와 같다")

print("=" * 60)
print("📥 Stop 토큰 바인딩 체인 실행")
print("=" * 60)
print(result_stop)
print()
print("💡 'SOLUTION' 토큰에서 생성이 중단되어 방정식만 출력됨")


📥 Stop 토큰 바인딩 체인 실행
EQUATION: \( x^3 + 7 = 12 \)



💡 'SOLUTION' 토큰에서 생성이 중단되어 방정식만 출력됨


## 2. OpenAI 함수 연결

### Tool Calling(도구 호출)이란?

OpenAI의 Tool Calling은 LLM이 외부 함수를 "호출할 수 있게" 해주는 기능이에요. LLM이 직접 실행하는 게 아니라, 어떤 함수를 어떤 인자로 호출해야 하는지 JSON으로 알려주는 거예요.

> 🔑 **핵심 개념**: Tool Calling은 원래 Function Calling이라는 이름으로 시작했지만, OpenAI가 여러 함수를 동시에 호출하는 기능을 추가하면서 Tools API로 확장됐어요. 현재 Function Calling은 deprecated(사용 중단 예정)이고, Tools API가 표준이에요. 두 용어가 혼용되지만 개념은 동일해요.

동작 과정을 정리하면 이래요:
1. 개발자가 함수 스키마(이름, 설명, 파라미터)를 JSON Schema 형식으로 정의해요
2. `bind(tools=[...])`로 모델에 전달해요
3. 모델이 사용자 질문을 분석하고, 함수 호출이 필요하다고 판단하면 함수명과 인자를 JSON으로 반환해요
4. 개발자가 해당 함수를 실제로 실행하고 결과를 다시 모델에 전달해요

> ⚠️ **주의**: `tool_choice`를 사용하지 않으면 모델이 함수를 호출할지 여부를 스스로 결정해요. 응답을 반드시 구조화된 형태로 받고 싶을 때는 `tool_choice`를 추가하세요.

> 🎯 **강의 포인트**: `bind(tools=[...])`를 사용하면 모델 응답이 `AIMessage.tool_calls` 형태로 반환돼요. 일반 텍스트 응답(`result.content`)과 다르므로, 후처리 시 `tool_calls` 속성을 확인해야 해요.

In [5]:
# ============================================================
# TODO: model.bind(tools=[solver_tool])로 OpenAI 함수를 바인딩하고 구조화된 응답을 받으세요
# 힌트: tools=[{"type": "function", "function": {...}}] 형식으로 정의
# 예상 결과: AIMessage.tool_calls에 {"name": "solver", "args": {...}} 포함
# ============================================================

# ---------------------------------------------------
# bind(tools=[...]): OpenAI Function Calling — 구조화된 응답 유도
# ---------------------------------------------------

# 1단계: OpenAI 함수(툴) 스키마 정의
# parameters: JSON Schema 형식으로 함수 입력 정의
openai_function = {
    "name": "solver",
    "description": "방정식을 수립하고 해결합니다",
    "parameters": {
        "type": "object",
        "properties": {
            "equation": {
                "type": "string",
                "description": "방정식의 대수식 표현",
            },
            "solution": {
                "type": "string",
                "description": "방정식의 해답",
            },
        },
        "required": ["equation", "solution"],
    },
}

solver_tool = {
    "type": "function",
    "function": openai_function,
}

# 2단계: 툴을 바인딩한 모델 생성
# bind(tools=[...]): 체인 구성 시점에 툴 목록을 고정
# - tool_choice를 추가하면 특정 함수 호출 강제 가능
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "대수 기호를 사용하여 다음 방정식을 작성한 다음 해결하세요.",
        ),
        ("human", "{equation_statement}"),
    ]
)

model_with_function = ChatOpenAI(model_name="gpt-4o-mini", temperature=0).bind(
    tools=[solver_tool],
    # tool_choice={"type": "function", "function": {"name": "solver"}},  # 강제 호출 시 활성화
)

chain_with_function = {"equation_statement": RunnablePassthrough()} | prompt | model_with_function

print("=" * 60)
print("✅ OpenAI 함수 바인딩 완료 (모델 선택)")
print("=" * 60)


✅ OpenAI 함수 바인딩 완료 (모델 선택)


In [6]:
# Functions 바인딩 체인 실행
result = chain_with_function.invoke("x의 세제곱에 7을 더하면 12와 같다")

print("=" * 60)
print("📥 Functions 바인딩 체인 실행")
print("=" * 60)
print(f"응답 타입: {type(result)}")
print(f"함수 호출 수: {len(result.tool_calls) if hasattr(result, 'tool_calls') else 0}")
if hasattr(result, "tool_calls") and result.tool_calls:
    for call in result.tool_calls:
        args = call.get("args") or call.get("arguments")
        print(f"  - 이름: {call.get('name')} | 인자: {args}")
else:
    print("  (함수 호출 없음)")
print()
print("💡 기본적으로 모델이 필요하다고 판단할 때 함수를 호출하며, 목적에 따라 tool_choice를 추가해 강제할 수 있습니다")


📥 Functions 바인딩 체인 실행
응답 타입: <class 'langchain_core.messages.ai.AIMessage'>
함수 호출 수: 1
  - 이름: solver | 인자: {'equation': 'x^3 + 7 = 12', 'solution': 'x = 1'}

💡 기본적으로 모델이 필요하다고 판단할 때 함수를 호출하며, 목적에 따라 tool_choice를 추가해 강제할 수 있습니다


## 3. OpenAI Tools 바인딩

### Functions API에서 Tools API로

> ⚠️ **주의**: OpenAI의 Functions API는 deprecated(사용 중단 예정)됐어요. 새 프로젝트에서는 Tools API를 사용하세요.

위의 섹션 2와 이 섹션 3 모두 `bind(tools=[...])`을 사용하지만, Tools API의 핵심 차이는 **한 번의 호출로 여러 도구를 병렬 호출**할 수 있다는 점이에요.

### Functions API vs Tools API 비교

| | Functions API (deprecated) | Tools API (현재 표준) |
|---|---|---|
| 병렬 호출 | 한 번에 하나만 | 여러 도구 동시 호출 가능 |
| 파라미터명 | `functions=`, `function_call=` | `tools=`, `tool_choice=` |
| 상태 | deprecated | 현재 표준 |
| 응답 형식 | `function_call` 필드 | `tool_calls` 배열 |

> 🎯 **강의 포인트**: Tools API의 가장 큰 장점은 한 번의 LLM 호출로 여러 도구를 동시에 호출한다는 거예요. 아래 예제에서 서울/뉴욕/LA 세 도시를 한 번에 조회하는 것이 그 예시예요. API 호출 횟수를 줄여 비용을 절감할 수 있어요.

In [7]:
# ============================================================
# TODO: bind(tools=[...])로 날씨 조회 도구를 모델에 바인딩하세요
# 힌트: tools 리스트에 {"type": "function", "function": {...}} 형식으로 정의
# 예상 결과: AIMessage.tool_calls에 도시별 날씨 조회 함수 호출 정보 포함
# ============================================================

# ---------------------------------------------------
# bind(tools=[...]): Tools API — 여러 도구를 병렬 호출 가능
# ---------------------------------------------------

# 1단계: 날씨 조회 도구 정의
# Tools API: Function Calling과 달리 한 번의 호출로 여러 도구를 병렬 호출 가능
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "주어진 위치의 현재 날씨를 가져옵니다",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {
                        "type": "string",
                        "description": "도시와 주, 예: 서울, 대한민국",
                    },
                    "unit": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "온도 단위",
                    },
                },
                "required": ["location"],
            },
        },
    }
]

# 2단계: Tools를 바인딩한 모델
# bind(tools=[...]): 체인 구성 시점에 도구 목록을 고정
model_with_tools = ChatOpenAI(model_name="gpt-4o-mini").bind(tools=tools)

print("=" * 60)
print("✅ OpenAI Tools 바인딩 완료")
print("=" * 60)


✅ OpenAI Tools 바인딩 완료


In [8]:
# Tools 바인딩 모델 실행
result = model_with_tools.invoke("서울, 뉴욕, 로스앤젤레스의 현재 날씨에 대해 알려줘?")

print("=" * 60)
print("📥 Tools 바인딩 모델 실행")
print("=" * 60)
print(f"응답: {result.content}")
print()
if hasattr(result, 'tool_calls') and result.tool_calls:
    print("🔧 호출된 도구:")
    for tool_call in result.tool_calls:
        print(f"  - {tool_call.get('name', 'N/A')}: {tool_call.get('args', {})}")
else:
    print("💡 모델이 도구 호출을 결정할 수 있음")


📥 Tools 바인딩 모델 실행
응답: 

🔧 호출된 도구:
  - get_current_weather: {'location': '서울, 대한민국', 'unit': 'celsius'}
  - get_current_weather: {'location': '뉴욕, 미국', 'unit': 'celsius'}
  - get_current_weather: {'location': '로스앤젤레스, 미국', 'unit': 'celsius'}


## 정리

### bind() 메서드의 주요 활용 비교

| 활용 | 코드 예시 | 효과 |
|------|-----------|------|
| Stop 토큰 | `model.bind(stop="SOLUTION")` | 특정 토큰에서 생성 중단 |
| Tools API | `model.bind(tools=[tool_schema])` | 도구 스키마 연결, 다중 도구 병렬 호출 가능 |
| tool_choice | `model.bind(tools=[...], tool_choice={...})` | 특정 도구 호출 강제 |
| 기타 파라미터 | `model.bind(temperature=0, max_tokens=100)` | 모델 파라미터 고정 |

### Tool Calling의 전체 흐름

```
1. 도구 스키마 정의 → 2. bind(tools=[...]) → 3. invoke() 실행
→ 4. 모델이 tool_calls 반환 → 5. 개발자가 실제 함수 실행 → 6. 결과를 모델에 전달
```

### 핵심 요점

- `bind()`는 체인 구성 시점에 파라미터를 고정하여 실행마다 반복 전달하는 번거로움을 없애줘요
- Stop 토큰으로 출력 길이를 의미 단위로 정밀하게 제어할 수 있어요
- `bind(tools=[...])`로 모델에 도구 스키마를 연결하면, 모델이 구조화된 JSON으로 함수 호출 정보를 반환해요
- Tools API는 한 번의 호출로 여러 도구를 병렬 호출할 수 있어요 (Functions API는 deprecated)
- `tool_choice`를 추가하면 특정 함수 호출을 강제할 수 있어요

### 다음 노트북 예고

다음 노트북(`10-Fallbacks.ipynb`)에서는 기본 모델이나 체인이 실패했을 때 자동으로 대체 경로로 전환하는 폴백(Fallback) 패턴을 배워요. 안정적인 LLM 서비스를 구축하기 위한 필수 패턴이에요.